# Setup environment

In [2]:

%pip install -q "crewai[tools, agentops]==0.114.0"
%pip install -q python-dotenv tavily-python scrapegraph-py

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
from tavily import TavilyClient
from scrapegraph_py import Client
import agentops
import os
import json

load_dotenv()
agentsops_api_key = os.getenv("AGENTSOPS_API_KEY")
llm_api_key = os.getenv("GEMINI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")
scrapegraph_api_key = os.getenv("SCRAPEGRAPH_API_KEY")

agentops.init(
    api_key = agentsops_api_key,
    skip_auto_end_session = True,
    default_tags = ['crewai']
)

basic_llm = LLM(
    model = "gemini/gemini-3.6-flash",
    api_key = llm_api_key,
    temperature = 0
)

search_client = TavilyClient(api_key = tavily_api_key)
scrapegraph_client = Client(api_key = scrapegraph_api_key)

/var/folders/c6/mbh888717p16dymfmd6ytftm0000gn/T/ipykernel_31311/1975815213.py:31: DeprecationWarning: scrapegraph-py v1.x is deprecated and will be removed in a future release. Please upgrade to scrapegraph-py v2.x for the new API surface. See migration guide: https://docs.scrapegraphai.com/transition-from-v1-to-v2
  scrapegraph_client = Client(api_key = scrapegraph_api_key)


In [57]:
# test tavily search
search_client.search("What is the capital of France?")

{'query': 'What is the capital of France?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_France',
   'title': 'List of capitals of France',
   'content': 'This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944.\n\n## Chronology\n\n[edit] [...] Tours (10–13 June 1940), the city served as the temporary capital of France during World War II after the government fled Paris due to the German advance.\n Bordeaux (June 1940), the government was relocated from Paris to Tours then Bordeaux very briefly during World War II, when it became apparent that Paris would soon fall into German hands. [...] Algiers (1943–1944), the city was made the seat of Free France, to be closer to the war in Europe.\n Paris (1945–present day).',
   'score': 0.8875269,
   'raw_content': None,
   'id': 'b8fb50-00'},
  {'url': 'https://home.adelphi.edu/~ca19535/page%204

In [58]:
output_dir = "./ai-agent-output"
os.makedirs(output_dir, exist_ok=True)

# Setup Agents

## Get Queries Agent

In [59]:
no_keywords = 10

# example output from the agent
{
    "queries" : [
        "keyword1", "keyword2", "keyword3"
    ]
}

{'queries': ['keyword1', 'keyword2', 'keyword3']}

In [ ]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(...,   # for required field
                               description = "A list of suggested search queries to be passed to the search engine.",
                               min_length = 1,
                               max_length = no_keywords
                        )

search_queries_recommendation_agent = Agent(
    role = "Search Queries Recommendation Agent",
    goal = "\n".join([
        "To provide a list of suggested search queries to be passed to the search engine.",
        "The queries must be varied and looking for specific items."
    ]),
    backstory = "The agent is designed to help in looking for products by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm = basic_llm,
    verbose = True
)

search_queries_recommendation_task = Task(
    description = "\n".join([
        "NexaNova is looking to buy {product_name} at the best prices (value for a price strategy)",
        "The company target any of these websites to buy from: {websites_list}",
        "The company wants to reach all available proucts on the internet to be compared later in another stage.",
        "The stores must sell the product in {country_name}",
        "Generate at maximum {no_keywords} queries.",
        "The search keywords must be in {language} language.",
        "Search keywords must contains specific brands, types or technologies. Avoid general keywords.",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),
    expected_output = "A JSON object containing a list of suggested search queries.",
    output_json = SuggestedSearchQueries,
    output_file = os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent = search_queries_recommendation_agent
)

## Search Agent

In [ ]:
# example output from the agent
{
    "results" : [
        {
            "title" : "",
            "url" : "",
            "content" : "",
            "score" : 0.50,
            "search_query" : ""
        },
        {
            "title" : "",
            "url" : "",
            "content" : "",
            "score" : 0.60,
            "search_query" : ""
        }
    ]
}

{'results': [{'title': '',
   'url': '',
   'content': '',
   'score': 0.5,
   'search_query': ''},
  {'title': '', 'url': '', 'content': '', 'score': 0.6, 'search_query': ''}]}

In [62]:
class SignleSearchResult(BaseModel):
    title: str
    url: str = Field(..., title = "The page url")
    content: str
    score: float
    search_query: str

class AllSearchResults(BaseModel):
    results: List[SignleSearchResult]

@tool
def search_engine_tool(query: str) -> str:
    """Useful for search-based queries. Use this to find current information about any query related pages using a search engine."""
    return search_client.search(query)
    

search_engine_agent = Agent(
    role = "Search Engine Agent",
    goal = "To search for products based on the suggested search query",
    backstory = "The agent is designed to help in looking for products by searching for products based on the suggested search queries.",
    llm = basic_llm,
    verbose = True,
    tools = [search_engine_tool]
)

search_engine_task = Task(
    description="\n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any susbicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_threshold}) .",
        "The search results will be used to compare prices of products from different websites."
    ]),
    expected_output = "A JSON object containing the search results.",
    output_json = AllSearchResults,
    output_file = os.path.join(output_dir, "step_2_search_results.json"),
    agent = search_engine_agent
)

# Scraping Agent

In [ ]:
# example output from the agent
{
    "products" : [
        {
            "page_url" : "",
            "product_title" : "",
            "product_url" : "",
            "product_image_url" : "",
            "product_current_price" : "",
            "product_original_price" : "",
            "product_specs" : [
                {
                    "specification_name" : "",
                    "specification_value" : ""
                }
            ],
            "agent_recommendation_rank" : "",
            "agent_recommendation_notes" : ""
        },
    ]
}

In [ ]:
class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str

class SingleExtractedProduct(BaseModel):
    page_url: str = Field(..., title = "The original url of the product page")
    product_title: str = Field(..., title = "The title of the product")
    product_image_url: str = Field(..., title = "The url of the product image")
    product_url: str = Field(..., title = "The url of the product")
    product_current_price: float = Field(..., title = "The current price of the product")
    product_original_price: float = Field(title = "The 1original price of the product before discount. Set to None if no discount", default = None)
    product_discount_percentage: float = Field(title = "The discount percentage of the product. Set to None if no discount", default = None)

    product_specs: List[ProductSpec] = Field(..., title = "The specifications of the product. Focus on the most important specs to compare.", min_length = 1, max_length = 5)

    agent_recommendation_rank: int = Field(..., title = "The rank of the product to be considered in the final procurement report. (out of 5, Higher is Better) in the recommendation list ordering from the best to the worst")
    agent_recommendation_notes: List[str]  = Field(..., title = "A set of notes why would you recommend or not recommend this product to the company, compared to other products.")


class AllExtractedProducts(BaseModel):
    products: List[SingleExtractedProduct]

@tool
def web_scraping_tool(page_url: str):
    """
    An AI Tool to help an agent to scrape a web page

    Example:
    web_scraping_tool(
        page_url = "https://www.noon.com/egypt-en/15-bar-fully-automatic-espresso-machine-1-8-l-1500"
    )
    """
    details = scrapegraph_client.smartscraper(
        website_url = page_url,
        user_prompt = "Extract ```json\n" + SingleExtractedProduct.schema_json() + "```\n From the web page"
    )

    return {
        "page_url" : page_url,
        "details" : details
    }

scrapegraph_agent = Agent(
    role = "Web scraping agent",
    goal = "To extract details from any website",
    backstory = "The agent is designed to help in looking for required values from any website url. These details will be used to decide which best product to buy.",
    llm = basic_llm,
    tools = [web_scraping_tool],
    verbose = True
)

scrapegraph_task = Task(
    description = "\n".join([
        "The task is to extract product details from any ecommerce store page url.",
        "The task has to collect results from multiple pages urls.",
        "Collect the best {top_recommendations_no} products from the search results."
    ]),
    expected_output = "A JSON object containing products details",
    output_json = AllExtractedProducts,
    output_file = os.path.join(output_dir, "step_3_search_results.json"),
    agent = scrapegraph_agent
)

# Generate Report Agent

In [ ]:
procurement_report_author_agent = Agent(
    role = "Procurement Report Author Agent",
    goal = "To generate a professional HTML page for the procurement report",
    backstory = "The agent is designed to assist in generating a professional HTML page for the procurement report after looking into a list of products.",
    llm = basic_llm,
    verbose = True
)

procurement_report_author_task = Task(
    description = "\n".join([
        "The task is to generate a professional HTML page for the procurement report.",
        "You have to use Bootstrap CSS framework for a better UI.",
        "Use the provided context about the company to make a specialized report.",
        "The report will include the search results and prices of products from different websites.",
        "The report should be structured with the following sections:",
        "1. Executive Summary: A brief overview of the procurement process and key findings.",
        "2. Introduction: An introduction to the purpose and scope of the report.",
        "3. Methodology: A description of the methods used to gather and compare prices.",
        "4. Findings: Detailed comparison of prices from different websites, including tables and charts.",
        "5. Analysis: An analysis of the findings, highlighting any significant trends or observations.",
        "6. Recommendations: Suggestions for procurement based on the analysis.",
        "7. Conclusion: A summary of the report and final thoughts.",
        "8. Appendices: Any additional information, such as raw data or supplementary materials."
    ]),
    expected_output = "A professional HTML page for the procurement report.",
    output_file = os.path.join(output_dir, "step_4_procurement_report.html"),
    agent = procurement_report_author_agent
)

# Run Crew AI

In [ ]:
about_company = "NexaNova is a company that provides AI solutions to help websites refine their search and recommendation systems."

company_context = StringKnowledgeSource(content = about_company)

In [ ]:
NexaNova_crew = Crew(
    agents = [
        search_queries_recommendation_agent,
        search_engine_agent,
        scrapegraph_agent,
        procurement_report_author_agent
        ],
    tasks = [
        search_queries_recommendation_task,
        search_engine_task,
        scrapegraph_task,
        procurement_report_author_task
    ],
    process = Process.sequential,
    knowledge_sources = [company_context]
)

In [ ]:
crew_results = NexaNova_crew.kickoff(
    inputs = {
        "product_name": "coffee machine for the office",
        "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keywords": 10,
        "language": "English",
        "score_threshold": 0.10,
        "top_recommendations_no": 10
    }
)

# Agent: Search Queries Recommendation Agent
## Task: Rankyx is looking to buy coffee machine for the office at the best prices (value for a price strategy)
The company target any of these websites to buy from: ['www.amazon.eg', 'www.jumia.com.eg', 'www.noon.com/egypt-en']
The company wants to reach all available proucts on the internet to be compared later in another stage.
The stores must sell the product in Egypt
Generate at maximum 10 queries.
The search keywords must be in English language.
Search keywords must contains specific brands, types or technologies. Avoid general keywords.
The search query must reach an ecommerce webpage for product, and not a blog or listing page.


# Agent: Search Queries Recommendation Agent
## Final Answer: 
{
  "queries": [
    "site:amazon.eg DeLonghi Magnifica S ECAM22.110.B Automatic Espresso Machine",
    "site:noon.com/egypt-en Philips 2200 Series Fully Automatic Espresso Machine EP2220/10",
    "site:jumia.com.eg Black+Decker 1.25L Drip Coffee